# Attention latent injection run

该 Notebook 用于 Colab 冷启动: 挂载 Google Drive、拉取仓库、加载 SD3.5 Medium、读取真实 attention geometry 包、重建 attention-relative carrier, 在真实 latent callback 中执行 attention-relative update, 生成 paired images、真实 latent update records 和质量指标, 并把所有核对文件打包到 Google Drive。

正式逻辑位于 `paper_workflow/colab_utils/attention_latent_injection.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 SD3.5 Medium 访问权限, 并配置 `HF_TOKEN`。
3. 确认 Drive 中已有 attention geometry 包, 默认查找 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `attention_geometry/attention_geometry_package_*.zip``。
4. 默认输出目录为 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `attention_latent_injection``。


In [ ]:
SLM_WM_PAPER_RUN_NAME = "pilot_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
# 依赖诊断已收敛到 paper_workflow.colab_utils.dependency_check, 在仓库 checkout 后统一执行。


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="attention_latent_injection",
)
print(paper_run_environment)

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report
dependency_report = build_notebook_dependency_report("sd35_runtime")
print(dependency_report)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
from pathlib import Path

geometry_dir = Path(os.environ['SLM_WM_ATTENTION_GEOMETRY_DRIVE_DIR'])
geometry_packages = sorted(geometry_dir.glob('attention_geometry_package_*.zip'))
assert geometry_packages, f'未找到 attention geometry 包: {geometry_dir}'
print({'geometry_package': str(geometry_packages[-1])})


In [ ]:
from paper_workflow.colab_utils.attention_latent_injection import run_default_attention_latent_injection_plan

attention_injection_summary = run_default_attention_latent_injection_plan(root='.')
assert attention_injection_summary['run_decision'] == 'pass', attention_injection_summary
assert attention_injection_summary['latent_update_count'] > 0, attention_injection_summary
assert attention_injection_summary['image_quality_metrics_ready'] is True, attention_injection_summary
attention_injection_summary


In [ ]:
from paper_workflow.colab_utils.notebook_entrypoint import package_workflow_outputs

archive_record = package_workflow_outputs(root='.', workflow_name="attention_latent_injection")
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/attention_latent_injection').glob('*.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/attention_latent_update').glob('*.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))
